# Africa country-mention network — SMWA extension

Professional reconstruction of the network section in the Social Media & Web Analytics submission. Nodes are African countries; directed edges represent one country mentioning another in a UN General Debate speech.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import load_ungd
from src.network_analysis import (
    AFRICAN_COUNTRIES, build_country_mention_network,
    centrality_table, strongest_edges,
)


## 1. Measurement audit

The **submitted coursework code searched for three-letter country codes inside speech text**. That is not a reliable named-entity rule because speeches normally contain country names rather than ISO3 codes.

The professional reconstruction therefore matches country names plus selected historical/orthographic aliases (e.g. *Zaire*, *Swaziland*, *Cape Verde*). This deliberately changes the measurement rule, so its centrality ranking is **not expected to reproduce the submitted network**.


In [ ]:
df = load_ungd(ROOT / "data" / "un-general-debates.csv")
G = build_country_mention_network(df)
print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())


## 2. Centrality and strongest ties


In [ ]:
centrality = centrality_table(G)
edges = strongest_edges(G, n=25)
centrality.head(15), edges.head(15)


In the submitted paper, Madagascar, Namibia, Comoros and Somalia are discussed as prominent and South Sudan as relatively isolated. Those are **source-reported findings from the original code**. The professional reconstruction should be interpreted on its own terms.


## 3. Clear network visualization


In [ ]:
top_edges = strongest_edges(G, n=70)
H = nx.DiGraph()
for row in top_edges.itertuples():
    H.add_edge(row.source, row.target, weight=row.weight)

pos = nx.spring_layout(H, seed=42, k=1.3)
weights = [H[u][v]["weight"] for u, v in H.edges()]
max_weight = max(weights)

fig, ax = plt.subplots(figsize=(14, 11))
nx.draw_networkx_nodes(
    H, pos,
    node_size=[140 + 18 * H.degree(n) for n in H],
    alpha=0.9, ax=ax,
)
nx.draw_networkx_edges(
    H, pos,
    width=[0.5 + 3 * w / max_weight for w in weights],
    alpha=0.25, arrowsize=10, ax=ax,
)
nx.draw_networkx_labels(
    H, pos,
    labels={n: AFRICAN_COUNTRIES[n] for n in H},
    font_size=7, ax=ax,
)
ax.set_title("Professional reconstruction — 70 strongest country-mention ties", loc="left")
ax.axis("off")
plt.show()


## 4. Interpretation boundary

Mention centrality measures attention under a specified text-matching rule. It is **not** a causal measure of diplomatic influence, alliance strength, cooperation, or geopolitical importance. The distinction between the original network and this corrected reconstruction is central to the QA documentation.
